# wav2vec2 Fine-Tuning

This notebook is set up to run end-to-end on a Google Colab runtime.

When it detects Colab, it will:
- clone the GitHub repo into `/content/speech-emotion-directions`
- install any missing training dependencies
- download the official RAVDESS speech archive
- rebuild the metadata CSV with Colab file paths
- launch the `facebook/wav2vec2-base` fine-tuning run

When it is run locally, it uses the current project folder instead.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
RAVDESS_URL = "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
RAVDESS_ARCHIVE_NAME = "Audio_Speech_Actors_01-24.zip"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)



def ensure_colab_packages() -> None:
    if not IS_COLAB:
        return

    package_map = {
        "yaml": "pyyaml",
        "pandas": "pandas",
        "numpy": "numpy",
        "soundfile": "soundfile",
        "librosa": "librosa",
        "torch": "torch",
        "torchaudio": "torchaudio",
        "transformers": "transformers",
        "datasets": "datasets",
        "accelerate": "accelerate",
        "sklearn": "scikit-learn",
        "tqdm": "tqdm",
    }
    missing_packages = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing_packages:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("Required training packages are already available.")



def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "FinalProject"]

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root locally. Expected a folder containing configs/wav2vec.yaml and src/."
    )


ensure_colab_packages()

if IS_COLAB:
    PROJECT_ROOT = Path("/content") / REPO_NAME
    if PROJECT_ROOT.exists() and (PROJECT_ROOT / ".git").exists():
        run_command(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"])
    elif PROJECT_ROOT.exists():
        print(f"Using existing project directory: {PROJECT_ROOT}")
    else:
        run_command(["git", "clone", REPO_URL, str(PROJECT_ROOT)])
else:
    PROJECT_ROOT = find_local_project_root()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd().resolve())

In [ ]:
import pandas as pd
from IPython.display import display

from src.data.ravdess_metadata import build_ravdess_metadata, save_metadata

raw_root = PROJECT_ROOT / "data" / "raw"
extract_dir = raw_root / "ravdess_audio_speech"
archive_path = raw_root / RAVDESS_ARCHIVE_NAME
metadata_path = PROJECT_ROOT / "data" / "metadata" / "ravdess_metadata.csv"

raw_root.mkdir(parents=True, exist_ok=True)
metadata_path.parent.mkdir(parents=True, exist_ok=True)



def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as response, destination.open("wb") as handle:
        shutil.copyfileobj(response, handle)



def count_wavs(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for _ in root.rglob("*.wav"))


wav_count = count_wavs(extract_dir)
print("Existing extracted wav files:", wav_count)

if wav_count < 1440:
    if not archive_path.exists():
        print("Downloading the official RAVDESS speech archive...")
        download_file(RAVDESS_URL, archive_path)
    else:
        print(f"Using existing archive: {archive_path}")

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    print("Extracting archive...")
    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(extract_dir)

wav_count = count_wavs(extract_dir)
print("Extracted wav files:", wav_count)
assert wav_count == 1440, f"Expected 1440 speech wav files, found {wav_count}."

df = build_ravdess_metadata(extract_dir)
save_metadata(df, metadata_path)

project_df = df[df["keep_for_project"]].copy()
print("Metadata saved to:", metadata_path)
print("Total speech clips:", len(df))
print("Project clips:", len(project_df))
display(project_df["final_label"].value_counts().sort_index().rename("count").to_frame())
display(project_df.groupby(["split", "final_label"]).size().unstack(fill_value=0))

In [ ]:
import json

import torch

from src.utils.config import load_yaml_config

config_path = PROJECT_ROOT / "configs" / "wav2vec.yaml"
config = load_yaml_config(config_path)

config["metadata_path"] = str(metadata_path.resolve())
config["output_dir"] = str((PROJECT_ROOT / config["output_dir"]).resolve())

if IS_COLAB and torch.cuda.is_available():
    config["batch_size"] = 8
    config["eval_batch_size"] = 16
    config["num_workers"] = 2

output_dir = Path(config["output_dir"])
output_dir.parent.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device_summary = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_summary = "Apple Metal (MPS)"
else:
    device_summary = "CPU"

assert config_path.exists(), f"Config file not found: {config_path}"
assert metadata_path.exists(), f"Metadata file not found: {metadata_path}"

print("Config file:", config_path)
print("Metadata file:", metadata_path)
print("Output directory:", output_dir)
print("Detected device:", device_summary)
print(json.dumps(config, indent=2))

In [ ]:
from src.training.train_wav2vec import run_training

run_training(config, dry_run=False)

In [ ]:
import json

print("Output directory:", output_dir)
print("Exists:", output_dir.exists())

if output_dir.exists():
    for path in sorted(output_dir.iterdir()):
        print(path.name)

for metrics_name in ["val_metrics.json", "test_metrics.json"]:
    metrics_path = output_dir / metrics_name
    if metrics_path.exists():
        with metrics_path.open("r", encoding="utf-8") as handle:
            metrics = json.load(handle)
        summary = {
            "accuracy": metrics.get("accuracy"),
            "macro_f1": metrics.get("macro_f1"),
            "weighted_f1": metrics.get("weighted_f1"),
        }
        print(f"\n{metrics_name}")
        print(json.dumps(summary, indent=2))